<div dir="rtl" align="right">

# تقييمُ SNR قبلَ وبعدَ ICA

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نَحسبُ نسبةَ الإشارةِ إلى الضوضاءِ (SNR) لِأربعِ قنواتٍ قبلَ وبعدَ تنظيفِ ICA. نُعرّفُ الإشارةَ كَقدرةِ نطاقِ ألفا (8-13 Hz) والضوضاءَ كَقدرةِ ما عداه.

## المُخرجاتُ المُتوقّعةُ

- مخططٌ شريطيٌّ مُجمّعٌ بأربعِ مجموعاتٍ (قنواتٌ)
- كلُّ مجموعةٍ تَحوي شريطين: قبلَ ICA (أزرق) وبعدَه (برتقالي)
- ارتفاعُ SNR بعدَ التنظيفِ يَدلُّ على فعاليّتِه

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| النطاقُ | 8-13 Hz | نطاقُ ألفا |
| nperseg | 1024 | نافذةُ ويلش |
| n_components | 4 | مكوناتُ ICA |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2.

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. حسابُ SNR قبلَ وبعدَ التنظيفِ

نُطبّقُ ICA ونَستبعدُ المكوّنَ الأكثرَ تبايناً، ثمّ نَحسبُ SNR لِكلِّ قناةٍ قبلَ وبعدَ التنظيفِ.

</div>

In [ ]:
import mne
from scipy.signal import welch

info = mne.create_info(ch_names, sfreq=fs, ch_types='eeg')
raw = mne.io.RawArray(eeg_data.T * 1e-6, info, verbose=False)

# NOTE: ICA works best with more channels than components.
ica = mne.preprocessing.ICA(
    n_components=3, random_state=97, max_iter=800, verbose=False
)
ica.fit(raw, verbose=False)

component_variances = np.var(ica.get_sources(raw).get_data(), axis=1)
exclude_idx = int(np.argmax(component_variances))
ica.exclude = [exclude_idx]
cleaned_raw = ica.apply(raw.copy(), verbose=False)
cleaned_data = cleaned_raw.get_data() * 1e6

def compute_snr(data, fs, fmin=8, fmax=13):
    freqs, psd = welch(data, fs=fs, nperseg=1024)
    signal_mask = (freqs >= fmin) & (freqs <= fmax)
    noise_mask = (freqs >= 0.5) & (freqs <= 80) & ~signal_mask
    signal_power = np.trapezoid(psd[signal_mask], freqs[signal_mask])
    noise_power = np.trapezoid(psd[noise_mask], freqs[noise_mask])
    if noise_power == 0:
        return 0.0
    return 10 * np.log10(signal_power / noise_power)

snr_before = [compute_snr(eeg_data[:, i], fs) for i in range(4)]
snr_after = [compute_snr(cleaned_data[i], fs) for i in range(4)]
for i, name in enumerate(ch_names):
    print(f'{name}: Before={snr_before[i]:.2f} dB, After={snr_after[i]:.2f} dB')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- كلُّ قناةٍ تَحوي شريطين: قبلَ (أزرق) وبعدَ (برتقالي) ICA
- ارتفاعُ الشريطِ البرتقاليِّ يَدلُّ على تحسّنِ SNR
- القنواتُ الأكثرُ تَأثّراً بِالآثارِ تَستفيدُ أكثر


</div>

In [ ]:
import plotly.graph_objects as go

x = list(ch_names)
fig = go.Figure()
fig.add_trace(go.Bar(name='Before ICA', x=x, y=snr_before,
                     marker_color='steelblue'))
fig.add_trace(go.Bar(name='After ICA', x=x, y=snr_after,
                     marker_color='orange'))
fig.update_layout(barmode='group', height=500,
                  title='SNR Evaluation - Before vs After ICA Cleaning',
                  xaxis_title='Channel', yaxis_title='SNR (dB)')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- SNR مقياسٌ كمّيٌّ يُقارنُ قدرةَ الإشارةِ بِقدرةِ الضوضاء
- ارتفاعُ SNR بعدَ التنظيفِ يَدلُّ على فعاليّتِه
- تعريفُ الإشارةِ والضوضاءِ يُؤثّرُ على القيمِ
- يُكمّلُ التقييمَ البصريَّ عبرَ FFT


</div>